<a href="https://colab.research.google.com/github/yevgenm/face_recognition/blob/main/facial_recognition_ai_python_with_comments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the notebook from the Programming Historian lesson by Charles Goldberg and Zach Haala, with comments added to the code: https://doi.org/10.46430/phen0119

The data link in the original lesson no longer works (November 2026). Instead, download `bethel_yearbooks.zip` from Brightspace, open the file browser on the left (the folder icon), and drag the zip file into it. Wait until the upload has finished before you run the first cell. The zip file contains six yearbooks from the Bethel University Digital Library, licensed CC BY-NC 3.0: https://cdm16120.contentdm.oclc.org/digital/collection/p16120coll2

We also removed a library that the code does not use, and added cells that show what the code does (the cells with a title that starts with "Look").

Run the cells from top to bottom. The "Look" cells only show results, so you can run them as often as you like. The other cells move between folders and delete files, so run each of them only once. To start again, choose Runtime > Restart session, run `!rm -rf /content/yearbook` in a new cell, and then run the notebook from the top. Your uploaded zip file stays where it is.

## Preliminaries

In [ ]:
%mkdir yearbook
%cd yearbook
# Create a folder called 'yearbook' and go there.

!unzip -q /content/bethel_yearbooks.zip -d .
# Unzip the yearbooks (1911-1961) into this folder. Upload bethel_yearbooks.zip from Brightspace first (see above).

%mkdir images

!pip install -q PyMuPDF DeepFace
import tensorflow as tf
!pip install -q "tf-keras=={tf.__version__.rsplit('.', 1)[0]}.*"
# Install PyMuPDF (for PDF files) and DeepFace (for face analysis), and the version of tf-keras that DeepFace
# needs with this version of TensorFlow. This can take a minute.

import os, shutil, fitz, cv2, numpy as np, pandas as pd
from deepface import DeepFace
from IPython.display import Image, display
# You may see a warning that the fitz API is deprecated: you can ignore it.

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
# Don't use a GPU: in Colab, DeepFace gives errors on a GPU.

yearbook_folder = os.getcwd()
# Remember the full path of the 'yearbook' folder, for the "Look" cells below.
print(yearbook_folder)

## PDF conversion

This cell turns every page of every yearbook into an image (a PNG file).

In [ ]:
path = './'
pdfs = [f for f in os.listdir(path) if f.endswith('.pdf')]
# A list of all PDF files in the current folder.

for pdf in pdfs:
    os.chdir(os.path.join('./images'))
    os.mkdir((pdf.split(".")[0]))
    newdir = (os.path.join('./images/' + os.path.join(pdf.split(".")[0])))
    os.chdir("..")
    # Create a folder in 'images' with the name of the PDF without '.pdf', for example images/1911.

    print("Now copying images into " + (newdir))
    shutil.copy(pdf, newdir)
    os.chdir(newdir)
    # Copy the PDF into that folder and go there.

    doc = fitz.open(pdf)
    # Open the PDF with PyMuPDF (in Python, the library is called fitz).

    for page in doc:
        pix = page.get_pixmap()
        # Turn the page into an image. By default, the resolution is low (72 dots per inch),
        # which makes small faces hard to find.

        pix.save("page-%i.png" % page.number)
        # Save the image as page-0.png, page-1.png, and so on.

    os.chdir(os.path.dirname(os.getcwd()))
    os.chdir("..")
    # Go back up two levels, to the 'yearbook' folder.

### Look: the page images

In [ ]:
for year in ['1911', '1921', '1931', '1941', '1951', '1961']:
    files = os.listdir(yearbook_folder + '/images/' + year)
    pages = [f for f in files if f.endswith('.png')]
    print(year, len(pages), 'pages')

display(Image(yearbook_folder + '/images/1911/page-15.png'))

## Object detection: one page

Before the code runs on all pages, try the face detector on the page above. Each detected face is a rectangle: `x` and `y` are its top left corner, and `w` and `h` are its width and height.

The detector has two settings. `scaleFactor` is how much the image is made smaller at each step of the search: 1.9 is a big step, so the search is fast, but it misses faces of some sizes. `minNeighbors` is how many overlapping detections are needed to accept a face: a higher number gives fewer false faces, but more missed faces. Try other values (for example `scaleFactor=1.1`) and run the cell again. The code below uses 1.9 and 4, like the lesson.

In [ ]:
image = cv2.imread(yearbook_folder + '/images/1911/page-15.png')
greyscale_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
# Read the page and make a greyscale copy: the detector works on greyscale images.

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
# Load a Haar cascade: a face detection model that comes with OpenCV.

detected_faces = face_cascade.detectMultiScale(image=greyscale_image, scaleFactor=1.9, minNeighbors=4)
print(len(detected_faces), 'faces found')
print(detected_faces)
# One row per face: x, y, w, h.

for (x, y, w, h) in detected_faces:
    cv2.rectangle(image, (x, y), (x+w, y+h), (255, 0, 0), 2)
# Draw a rectangle around each face. OpenCV uses the colour order blue, green, red, so (255, 0, 0) is blue.

cv2.imwrite(yearbook_folder + '/test_page.png', image)
display(Image(yearbook_folder + '/test_page.png'))

## Object detection: all pages

This cell finds the faces on all pages, saves each face as a separate image in a 'faces' folder for each year, and then deletes the page images. Run the cells above first.

In [ ]:
path = './'
os.chdir(os.path.join(path + 'images'))
dirs = os.listdir(path)
# Go to the 'images' folder and list its contents: one folder per yearbook.

for dir in dirs:
    os.chdir(os.path.join(path + dir))
    pngs = [f for f in os.listdir(path) if f.endswith('.png')]
    # Go to the folder of one yearbook and list its page images.

    if not os.path.exists((dir) + ' faces'):
    # If there is a 'faces' folder already (because you ran this cell before), this yearbook is skipped.

        print("New 'faces' directory created in " + (dir) + " folder")
        os.makedirs((dir) + ' faces')
        count = 0

        for png in pngs:
            image = cv2.imread(png)
            greyscale_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
            detected_faces = face_cascade.detectMultiScale(image=greyscale_image, scaleFactor=1.9, minNeighbors=4)
            # The same steps as in the cell above, for each page.

            count = 0
            # The faces on each page are numbered 1, 2, 3...

            for (x, y, w, h) in detected_faces:
                try:
                # If something goes wrong in the 'try' part, Python continues with the 'except' part
                # instead of stopping.

                    xpadding = 20
                    ypadding = 40
                    crop_face = image[y-ypadding : y+h+ypadding, x-xpadding : x+w+xpadding]
                    # Cut out the face with some space around it. image[rows, columns]: rows go from top
                    # to bottom (y), columns from left to right (x). If a face is close to the edge of the
                    # page, y-ypadding or x-xpadding is negative, and the cut-out goes wrong.

                    count += 1
                    face = cv2.rectangle(crop_face, (x, y), (x+w, y+h), (255, 0, 0), 2)
                    cv2.imwrite(path + (dir) + ' faces/' + str(count) + '_' + png, face)
                    # Draw a rectangle and save the face, for example '1951 faces/1_page-10.png'.

                except (Exception):
                    print("An error happened")
                    continue

            os.remove(os.path.join(path, png))
            # Delete the page image. After this cell, only the PDF and the faces are left.

    os.chdir("..")

### Look: the faces

In [ ]:
for year in ['1911', '1921', '1931', '1941', '1951', '1961']:
    faces = os.listdir(yearbook_folder + '/images/' + year + '/' + year + ' faces')
    print(year, len(faces), 'faces')

faces_1951 = os.listdir(yearbook_folder + '/images/1951/1951 faces')
for face in faces_1951[:5]:
    print(face)
    display(Image(yearbook_folder + '/images/1951/1951 faces/' + face))

## Identify smiles: one face

DeepFace guesses the emotion on a face. The result is a list with one dictionary for each face it finds. The dictionary has a score for each of seven emotions (they add up to 100), the emotion with the highest score (`dominant_emotion`), and where DeepFace found the face (`region`). Change the file name to try other faces.

In [ ]:
result = DeepFace.analyze(yearbook_folder + '/images/1951/1951 faces/1_page-10.png', actions=['emotion'])
print(result)

## Identify smiles: all faces

The first cell makes a list of all faces, the second one runs DeepFace on each face, and the third one makes a table with the share of smiling faces per year.

In [ ]:
%cd ..
# Go back to the 'yearbook' folder.

pngs = []
file_count = 0
file_count_list = []
years = ['1911', '1921', '1931', '1941', '1951', '1961']

for year in years:
    path = './images' + '/' + year

    for root, dirs, files in os.walk(path):
        for dir in dirs:
        # os.walk goes through all folders inside images/1911 and so on. The only folder there is '1911 faces'.

            path = path + '/' + (year + ' faces')
            if (file_count != 0):
                file_count_list.append(file_count)
            file_count = 0

            for f in os.listdir(path):
                if f.endswith('.png'):
                    pngs.append(path + '/' + f)
                    file_count = file_count + 1

file_count_list.append(file_count)
# pngs: one long list with the faces of all years. file_count_list: the number of faces per year.

print(len(pngs), 'faces in total')
print(file_count_list)

In [ ]:
number_smiles = 0
number_nonsmiles = 0
num_errors = 0
smile_counts = []
nonsmile_counts = []
error_counts = []
# Counters for the current year, and lists for the results of each year.

total_loops = 0
count = 0
iterator = 0
# count: the number of faces so far in the current year. iterator: which year we are in (0 is 1911).

for png in pngs:
    try:
        total_loops = total_loops + 1
        count = count + 1

        if (count != (file_count_list[iterator] + 1)):
        # Check whether this face still belongs to the current year.

            demography = DeepFace.analyze(png, actions=['emotion'])
            print(png, demography[0]['dominant_emotion'])

            if(demography[0]['dominant_emotion'] == 'happy'):
                number_smiles = number_smiles + 1
            else:
                number_nonsmiles = number_nonsmiles + 1
            # 'happy' counts as a smile, all other emotions as no smile.

        else:
        # The current year is finished: save its results and start the next year.

            count = count - 1
            smile_counts.append(number_smiles / count)
            nonsmile_counts.append(number_nonsmiles / count)
            error_counts.append(num_errors / count)
            number_smiles = 0
            number_nonsmiles = 0
            num_errors = 0
            iterator = iterator + 1
            count = 0

    except (Exception):
    # If DeepFace can't find a face in the image, it gives an error.
        num_errors = num_errors + 1
        print("An error happened")
        continue

smile_counts.append(number_smiles / count)
nonsmile_counts.append(number_nonsmiles / count)
error_counts.append(num_errors / count)
# Save the results of the last year.

print(count)
# The number of faces counted for the last year (1961).

In [ ]:
dict = {'Years': years,
        'Smiles': smile_counts,
        'Non-Smiles': nonsmile_counts,
        "Error Weight": error_counts}
# (dict and dir, used above, are also the names of built-in Python functions, so it is better not to use
# them as variable names.)

data = pd.DataFrame(dict)
data.to_csv('YearbookOutput.csv', index=False)
data

## Download the results

In [ ]:
from google.colab import files
files.download('YearbookOutput.csv')
# Download the CSV file to your computer.